# Seasonal Periods in Time Series

This notebook explains **seasonal periods** using Markdown notes, formulas, Python examples, and plots.

## Learning goals

By the end, you should be able to:

- explain what a seasonal period is, and how seasonality differs from cycles and calendar effects;
- distinguish **data frequency** from **seasonal period**, and map both to pandas and forecasting libraries;
- calculate seasonal periods for common time intervals;
- see seasonality with a seasonal plot and separate it from trend with a simple decomposition;
- tell **additive** from **multiplicative** seasonality;
- understand multiple and non-integer seasonalities, and how much history each needs;
- use seasonal lags, seasonal differencing and lag correlations;
- build a **seasonal naive** forecast, the baseline every seasonal model must beat.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

## 1. What is a seasonal period?

A **seasonal period** is the number of observations before a seasonal pattern repeats.

If the seasonal period is $m$, observations separated by $m$ steps may behave similarly.

Examples:

- monthly data with yearly seasonality: $m=12$
- daily data with weekly seasonality: $m=7$
- hourly data with daily seasonality: $m=24$

> The seasonal period is measured in **number of observations**.

## 2. Seasonality vs cycles vs calendar effects

Not every repeating movement has a seasonal period.

| Pattern | Repeats after a **fixed, known** number of observations? | Example | Has an $m$? |
|---|---|---|---|
| **Seasonality** | Yes | Ice-cream sales peak every July | Yes |
| **Cycle** | No: length varies and is not known in advance | Economic booms and recessions lasting 2–10 years | No |
| **Moving calendar effect** | No: the date moves each year | Easter, Ramadan, Diwali, Chinese New Year | No |
| **Fixed calendar effect** | Tied to the calendar but uneven | Months have 28–31 days; months have 4 or 5 weekends | Partly |

Consequences:

- A seasonal period comes from the **calendar or the process** (shops close on Sunday, heating runs in winter), not from staring at a chart. You should be able to say *why* the pattern repeats.
- Cycles cannot be captured with a seasonal period; they need trend or external drivers.
- Moving holidays need explicit **event features** (for example `is_easter_week`), because lag 12 or lag 52 will point at the wrong dates.

## 3. Common seasonal periods

| Data frequency | Repeating pattern | Seasonal period |
|---|---|---:|
| Quarterly | Yearly | 4 |
| Monthly | Yearly | 12 |
| Weekly | Yearly | about 52 |
| Daily | Weekly | 7 |
| Daily | Yearly | about 365.25 |
| Hourly | Daily | 24 |
| Hourly | Weekly | 168 |
| Hourly | Yearly | 8766 |
| 30-minute | Daily | 48 |
| 15-minute | Daily | 96 |
| Minute | Hourly | 60 |
| Minute | Daily | 1440 |
| Minute | Weekly | 10080 |
| Minute | Yearly | 525960 |

A useful idea is:

$$
\text{seasonal period}
=
\frac{\text{length of seasonal cycle}}
{\text{time between observations}}
$$

Example: 15-minute data with a daily cycle → $\frac{24 \times 60}{15} = 96$.

## 4. Monthly data example

Suppose sales are recorded once per month and the pattern repeats every year.

There are 12 monthly observations in one year:

$$
m=12
$$


In [ ]:
dates = pd.date_range("2022-01-01", periods=48, freq="MS")
t = np.arange(48)

seasonal = 20 * np.sin(2 * np.pi * t / 12)
trend = 0.5 * t
noise = np.random.normal(0, 3, 48)

sales = 100 + trend + seasonal + noise

monthly_df = pd.DataFrame({
    "date": dates,
    "sales": sales
})

monthly_df.head()

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(monthly_df["date"], monthly_df["sales"], marker="o")
plt.title("Synthetic Monthly Sales")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.grid(alpha=0.3)
plt.show()

The pattern repeats approximately every 12 observations, although trend and noise prevent exact repetition.

## 5. Seeing seasonality: the seasonal plot

A line plot over time mixes trend, seasonality and noise. A **seasonal plot** puts the season on the x-axis (here: month) and draws **one line per cycle** (here: per year).

- If the lines share the same shape, the seasonality is stable.
- If the lines are shifted up or down, there is trend (the level changes between years).
- If the shape changes from year to year, the seasonality is evolving and older years matter less.

In [ ]:
seasonal_view = monthly_df.assign(
    year=monthly_df["date"].dt.year,
    month=monthly_df["date"].dt.month,
).pivot(index="month", columns="year", values="sales")

ax = seasonal_view.plot(figsize=(10, 4), marker="o")
ax.set_title("Seasonal Plot: One Line per Year")
ax.set_xlabel("Month")
ax.set_ylabel("Sales")
ax.set_xticks(range(1, 13))
ax.grid(alpha=0.3)
plt.show()

All four years have the same shape (peak around April, trough around October), and each year sits slightly higher than the previous one. That is stable yearly seasonality ($m=12$) plus a gentle upward trend.

## 6. Frequency vs seasonal period

**Frequency** tells us how often observations are recorded.

**Seasonal period** tells us how many observations form one repeating cycle.

For hourly data with a daily cycle:

$$
24 \text{ observations/day}
$$

so:

$$
m=24
$$

For hourly data with a weekly cycle:

$$
24\times7=168
$$

so:

$$
m=168
$$


In [ ]:
print("Hourly data, daily seasonality:", 24)
print("Hourly data, weekly seasonality:", 24 * 7)
print("Hourly data, yearly seasonality:", 24 * 365.25)

### Frequency and seasonal period in code

The two ideas appear as **separate settings** in pandas and forecasting libraries:

| Data | pandas `freq` alias | Typical seasonal period $m$ |
|---|---|---:|
| Monthly, dated at month start | `"MS"` | 12 |
| Monthly, dated at month end | `"ME"` | 12 |
| Quarterly, dated at quarter start | `"QS"` | 4 |
| Weekly (Sunday) | `"W"` | 52 |
| Daily | `"D"` | 7 |
| Hourly | `"h"` | 24 |
| 30-minute | `"30min"` | 48 |

Recent pandas versions use lowercase `"h"` and `"min"`; older code may show `"H"` and `"T"`.

Where $m$ goes in common libraries:

| Library | Frequency setting | Seasonal period setting |
|---|---|---|
| StatsForecast (`AutoETS`, `AutoARIMA`, ...) | `freq="MS"` | `season_length=12` |
| statsmodels `seasonal_decompose`, `STL` | taken from the index | `period=12` |
| statsmodels `ExponentialSmoothing` | taken from the index | `seasonal_periods=12` |

`freq` says **how far apart** the observations are; `m` says **how many of them** make one cycle. Neither one is the forecast horizon `h`.

pandas can tell you the frequency of a date column, but it **cannot** tell you the seasonal period:

In [ ]:
print("Monthly data frequency:", pd.infer_freq(monthly_df["date"]))
print("Daily data frequency:  ", pd.infer_freq(pd.date_range("2024-01-01", periods=10, freq="D")))

## 7. Daily data can have multiple seasonalities

Daily data may contain:

### Weekly seasonality

$$
m=7
$$

### Annual seasonality

$$
m\approx365.25
$$

So one time series can contain more than one repeating seasonal pattern.


In [ ]:
dates = pd.date_range("2024-01-01", periods=730, freq="D")
t = np.arange(len(dates))

weekly = 8 * np.sin(2 * np.pi * t / 7)
annual = 15 * np.sin(2 * np.pi * t / 365.25)
trend = 0.02 * t
noise = np.random.normal(0, 2.5, len(t))

daily_sales = 100 + trend + weekly + annual + noise

daily_df = pd.DataFrame({
    "date": dates,
    "sales": daily_sales
})

daily_df.head()

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(daily_df["date"], daily_df["sales"])
plt.title("Daily Data with Weekly and Annual Seasonality")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.grid(alpha=0.3)
plt.show()

Over two years, the long annual movement is visible. To make weekly repetition easier to see, zoom into a few weeks.

In [ ]:
sample = daily_df.iloc[:42]

plt.figure(figsize=(12, 4))
plt.plot(sample["date"], sample["sales"], marker="o")
plt.title("First 6 Weeks: Weekly Seasonality")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.show()

## 8. Minute-level data

If observations are recorded every minute:

- hourly seasonality: $60$
- daily seasonality: $60\times24=1440$
- weekly seasonality: $60\times24\times7=10080$
- yearly seasonality: $60\times24\times365.25=525960$

In [ ]:
minute_periods = pd.Series({
    "hourly": 60,
    "daily": 60 * 24,
    "weekly": 60 * 24 * 7,
    "yearly": 60 * 24 * 365.25
}, name="seasonal_period")

minute_periods

## 9. Why 52 weeks is only an approximation

An average year is approximately 365.25 days.

$$
\frac{365.25}{7}\approx52.18
$$

So yearly seasonality in weekly data is not exactly 52.

Many models require an integer seasonal period, so $52$ is often used as a practical approximation.


The same is true for daily data with a yearly cycle: $m \approx 365.25$ is not a whole number.

In [ ]:
weeks_per_year = 365.25 / 7

print("Average weeks per year:", weeks_per_year)
print("Rounded value:", round(weeks_per_year))

### Non-integer periods: Fourier terms

Seasonal models such as ETS and seasonal ARIMA need an **integer** $m$, and they also become slow or unstable when $m$ is large (for example $m=365$ on daily data).

The usual alternative is **Fourier terms**: pairs of sine and cosine features with the exact period, even when it is not a whole number:

$$
\sin\left(\frac{2\pi k t}{m}\right), \quad \cos\left(\frac{2\pi k t}{m}\right), \quad k = 1, 2, \dots, K
$$

- $m$ can be 365.25 or 52.18.
- $k=1$ gives one smooth wave per year; larger $k$ adds sharper detail.
- $K$ (the number of pairs) is chosen by validation; small values like 2–5 are common.

These columns are ordinary features, so any regression or machine-learning model can use them.

In [ ]:
def fourier_terms(t, period, K):
    features = {}
    for k in range(1, K + 1):
        features[f"sin_{k}"] = np.sin(2 * np.pi * k * t / period)
        features[f"cos_{k}"] = np.cos(2 * np.pi * k * t / period)
    return pd.DataFrame(features)


t_daily = np.arange(len(daily_df))
yearly_fourier = fourier_terms(t_daily, period=365.25, K=2)
yearly_fourier.index = daily_df["date"]

yearly_fourier.plot(figsize=(12, 4), title="Yearly Fourier Terms (period = 365.25 days, K = 2)")
plt.grid(alpha=0.3)
plt.show()

yearly_fourier.head()

## 10. How much history do you need?

To estimate a seasonal pattern, a model must see it repeat. A practical rule:

> Have **at least 2 full seasonal cycles** ($2m$ observations); 3 or more is better.

With fewer than two cycles, the model cannot tell seasonality apart from trend or a one-off event, because each season has been seen only once.

In [ ]:
history_needed = pd.DataFrame({
    "data": ["Monthly", "Quarterly", "Weekly", "Daily (weekly cycle)", "Daily (yearly cycle)", "Hourly (yearly cycle)"],
    "m": [12, 4, 52, 7, 365.25, 8766],
})
history_needed["minimum_observations (2m)"] = (2 * history_needed["m"]).round().astype(int)
history_needed["minimum_history"] = ["2 years", "2 years", "2 years", "2 weeks", "2 years", "2 years"]

history_needed

Notice that the **time span** needed depends on the cycle, not the frequency: any yearly seasonality needs about two years of history, whether you have 24 monthly points or 17,532 hourly points.

This is why new products often cannot use yearly seasonality yet: there is not enough history. Common workarounds are borrowing the pattern from similar products or from the product category.

## 11. Additive vs multiplicative seasonality

**Additive:** the seasonal swing is a **fixed number of units**, whatever the level.

$$
y_t = \text{trend}_t + \text{seasonal}_t + \text{noise}_t
$$

**Multiplicative:** the seasonal swing is a **fixed percentage** of the level, so it grows as the series grows.

$$
y_t = \text{trend}_t \times \text{seasonal}_t \times \text{noise}_t
$$

Retail sales are often multiplicative: a December peak of +30% means 30 extra units for a small shop and 30,000 for a large one.

In [ ]:
t48 = np.arange(48)
dates48 = pd.date_range("2022-01-01", periods=48, freq="MS")
level = 100 + 4 * t48

additive = level + 20 * np.sin(2 * np.pi * t48 / 12)
multiplicative = level * (1 + 0.2 * np.sin(2 * np.pi * t48 / 12))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(dates48, additive)
axes[0].set_title("Additive: constant swing")

axes[1].plot(dates48, multiplicative)
axes[1].set_title("Multiplicative: swing grows with level")

axes[2].plot(dates48, np.log(multiplicative))
axes[2].set_title("log(multiplicative): constant swing again")

for ax in axes:
    ax.grid(alpha=0.3)
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

How to choose:

- Look at the plot: if the peaks and troughs fan out as the level rises, think multiplicative.
- Taking the **log** turns multiplicative seasonality into additive seasonality, because $\log(a \times b) = \log a + \log b$. This is why log transforms are common in forecasting.
- Logs need strictly positive values, so they do not work directly on series with zeros, such as intermittent demand.
- ETS models have both versions; `AutoETS` can try each and choose by information criterion.

## 12. Separating trend and seasonality: classical decomposition

Decomposition splits a series into trend, seasonal and remainder parts. Doing it by hand once shows exactly where the seasonal period is used.

For monthly data ($m=12$), additive version:

1. **Trend:** a centred moving average over exactly one cycle. An even $m$ has no middle point, so we use a "2×12" moving average: 13 months, with half weight on the two end months.
2. **Detrend:** subtract the trend from the series.
3. **Seasonal:** average the detrended values for each month (all Januaries, all Februaries, ...). Then shift the 12 values so they sum to 0.
4. **Remainder:** what is left after removing trend and seasonal.

Libraries do this for you with `statsmodels.tsa.seasonal.seasonal_decompose(series, period=12)`, or the more robust `STL(series, period=12)`.

In [ ]:
m = 12
sales_series = monthly_df.set_index("date")["sales"]

# 1. Trend: centred 2x12 moving average (weights 1/24, 1/12 x 11, 1/24)
weights = np.r_[0.5, np.ones(m - 1), 0.5] / m
trend_component = sales_series.rolling(m + 1, center=True).apply(lambda x: np.dot(x, weights), raw=True)

# 2. Detrend
detrended = sales_series - trend_component

# 3. Seasonal: average by month, then centre so the 12 values sum to 0
month_effect = detrended.groupby(detrended.index.month).mean()
month_effect = month_effect - month_effect.mean()
seasonal_component = pd.Series(month_effect.loc[sales_series.index.month].to_numpy(), index=sales_series.index)

# 4. Remainder
remainder = sales_series - trend_component - seasonal_component

fig, axes = plt.subplots(4, 1, figsize=(11, 9), sharex=True)
for ax, (name, part) in zip(axes, [("Observed", sales_series), ("Trend", trend_component),
                                   ("Seasonal", seasonal_component), ("Remainder", remainder)]):
    ax.plot(part)
    ax.set_ylabel(name)
    ax.grid(alpha=0.3)
axes[0].set_title("Classical Additive Decomposition (m = 12)")
plt.tight_layout()
plt.show()

month_effect.round(1).rename("seasonal effect (units vs trend)")

The trend is missing for the first and last 6 months: a centred window needs 6 months on each side. The seasonal effects recover the sine wave we built into the data (about +20 in April, about −20 in October), and the remainder looks like the random noise we added.

## 13. Seasonal lags

A seasonal lag compares each observation with an observation one seasonal cycle earlier.

Examples:

- daily data with weekly seasonality → lag 7
- monthly data with yearly seasonality → lag 12
- hourly data with daily seasonality → lag 24

In pandas, use `.shift()`.

In [ ]:
example = pd.DataFrame({
    "value": [10, 12, 13, 9, 11, 18, 20, 11, 13, 14, 10, 12, 19, 21]
})

example["lag_7"] = example["value"].shift(7)
example

For daily data, `lag_7` aligns each day with the corresponding day one week earlier.

## 14. Seasonal differencing

A **seasonal difference** subtracts the value one cycle earlier:

$$
y_t - y_{t-m}
$$

For monthly data this is "change versus the same month last year". It removes a stable seasonal pattern, because each month is compared with the same month. ARIMA-type models use it to make seasonal data more stable (stationary).

Compare it with the ordinary difference $y_t - y_{t-1}$, which removes trend but keeps seasonality. The first $m$ values are missing, because there is no earlier cycle to compare with.

In [ ]:
monthly_df["sales_lag_12"] = monthly_df["sales"].shift(12)
monthly_df["difference_from_last_year"] = (
    monthly_df["sales"] - monthly_df["sales_lag_12"]
)

monthly_df.tail(18)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(monthly_df["date"], monthly_df["sales"])
axes[0].set_title("Original monthly sales")

axes[1].plot(monthly_df["date"], monthly_df["difference_from_last_year"], color="tab:orange")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("Seasonal difference: sales − sales 12 months earlier")

for ax in axes:
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

The repeating wave is gone. What is left is noise around about +6, which is the trend: $0.5 \times 12 = 6$ extra units per year.

## 15. Lag correlation

If a seasonal pattern is strong, the series may be strongly related to a lagged copy of itself.

For monthly yearly seasonality, lag 12 is especially interesting.

For daily weekly seasonality, lag 7 is especially interesting.

In [ ]:
def lag_correlation(series, lag):
    return series.corr(series.shift(lag))

print(
    "Monthly correlation at lag 12:",
    round(lag_correlation(monthly_df["sales"], 12), 3)
)

print(
    "Daily correlation at lag 7:",
    round(lag_correlation(daily_df["sales"], 7), 3)
)

In [ ]:
lags = range(1, 25)
corrs = [lag_correlation(monthly_df["sales"], lag) for lag in lags]

plt.figure(figsize=(10, 4))
plt.stem(list(lags), corrs)
plt.axvline(12, linestyle="--", label="Lag 12")
plt.title("Lag Correlations for Monthly Data")
plt.xlabel("Lag")
plt.ylabel("Correlation")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### Caution: trend inflates lag correlations

When a series has a trend, values that are close in time are similar just because the level is similar. That makes lag correlations high at many lags, which can hide or fake seasonality.

A common fix is to **remove the trend first**, for example with a first difference, and then look at the correlations again:

In [ ]:
trend_heavy = pd.Series(100 + 3 * np.arange(48) + 10 * np.sin(2 * np.pi * np.arange(48) / 12) + np.random.normal(0, 3, 48))

lags = range(1, 25)
raw_corrs = [lag_correlation(trend_heavy, lag) for lag in lags]
diff_corrs = [lag_correlation(trend_heavy.diff(), lag) for lag in lags]

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
axes[0].stem(list(lags), raw_corrs)
axes[0].set_title("Strong trend: lag correlations of the raw series")
axes[1].stem(list(lags), diff_corrs)
axes[1].set_title("Same series after first difference")

for ax in axes:
    ax.axvline(12, linestyle="--", color="tab:red", label="Lag 12")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Lag")
    ax.grid(alpha=0.3)
    ax.legend()
axes[0].set_ylabel("Correlation")
plt.tight_layout()
plt.show()

On the left, every lag looks strongly correlated, because of the trend. On the right, after differencing, the seasonal structure stands out: positive at lag 12 and negative around lag 6, half a cycle away.

pandas also has a built-in autocorrelation plot: `pd.plotting.autocorrelation_plot(series)`.

This is a simple diagnostic. In real time-series analysis, seasonality may also be studied using autocorrelation functions, decomposition methods, spectral methods, or model-based approaches.

## 16. Seasonal naive: the baseline every seasonal model must beat

The **seasonal naive** forecast repeats the value from the same season in the last observed cycle:

$$
\hat{y}_{T+h} = y_{T+h-m}, \quad h = 1, \dots, m
$$

For monthly data: next January = last January, next February = last February, and so on.

It needs only the seasonal period, no fitting. If a sophisticated model cannot beat seasonal naive on held-out data, the model is not adding value.

Below, the last 12 months are held out and three simple forecasts are compared.

In [ ]:
m = 12
train = monthly_df["sales"].iloc[:-m]
test = monthly_df["sales"].iloc[-m:]
test_dates = monthly_df["date"].iloc[-m:]

forecasts = pd.DataFrame({
    "actual": test.to_numpy(),
    "seasonal_naive": train.iloc[-m:].to_numpy(),            # same month last year
    "naive": np.repeat(train.iloc[-1], m),                    # last observed value
    "mean": np.repeat(train.mean(), m),                       # average of history
}, index=test_dates)

mae = (forecasts.drop(columns="actual").sub(forecasts["actual"], axis=0)).abs().mean().round(2)
print("Mean absolute error on the last 12 months:")
print(mae.sort_values().to_string())

ax = forecasts.plot(figsize=(11, 4), marker="o")
ax.set_title("Hold-out comparison: seasonal naive vs naive vs mean")
ax.set_ylabel("Sales")
ax.grid(alpha=0.3)
plt.show()

Seasonal naive wins easily here, because the data is strongly seasonal. It is not perfect: it copies last year's noise and ignores the trend (it runs about 6 units low), which is the gap better models such as ETS try to close.

## 17. Multiple seasonalities in hourly data

Hourly electricity demand might contain all of these:

| Pattern | Seasonal period |
|---|---:|
| Daily | 24 |
| Weekly | 168 |
| Yearly | 8766 |

This is called **multiple seasonality**.

Specialized models such as **MSTL**, **MFLES**, and **TBATS** can be useful for more complicated seasonal structures.

## 18. Helper function

In [ ]:
def common_seasonal_periods(frequency):
    periods = {
        "quarterly": {"yearly": 4},
        "monthly": {"yearly": 12},
        "weekly": {"yearly_approx": 52},
        "daily": {"weekly": 7, "yearly_approx": 365.25},
        "hourly": {
            "daily": 24,
            "weekly": 24 * 7,
            "yearly_approx": 24 * 365.25,
        },
        "minute": {
            "hourly": 60,
            "daily": 60 * 24,
            "weekly": 60 * 24 * 7,
            "yearly_approx": 60 * 24 * 365.25,
        },
    }

    frequency = frequency.lower()

    if frequency not in periods:
        raise ValueError(f"Choose from: {list(periods)}")

    return periods[frequency]


common_seasonal_periods("hourly")

## 19. Practice questions

### Question 1

You have monthly revenue data with a yearly cycle. What is $m$?

### Question 2

Website traffic is recorded hourly and has a daily pattern. What is $m$?

### Question 3

Electricity consumption is recorded every 30 minutes and has a daily pattern. What is $m$?

### Question 4

Daily sales show both weekday effects and yearly effects. Which seasonal periods might you consider?

### Question 5

Sensor readings arrive every 15 minutes and have a daily pattern. What is $m$?

### Question 6

A product launched 18 months ago and you have monthly sales. Can you reliably estimate yearly seasonality? Why?

### Question 7

Sales peak every year around Easter. Will a lag-52 feature on weekly data capture this reliably?

### Question 8

The December peak was 50 units when the business was small and 500 units now that it is ten times bigger. Additive or multiplicative?

In [ ]:
# Answers

q1 = 12
q2 = 24
q3 = 2 * 24
q4 = {"weekly": 7, "yearly_approx": 365.25}
q5 = 4 * 24
q6 = "No: 18 months is less than 2 full cycles (2 x 12 = 24 months)"
q7 = "No: Easter moves between March and April, so use an explicit holiday feature"
q8 = "Multiplicative: the swing grows in proportion to the level"

for i, answer in enumerate([q1, q2, q3, q4, q5, q6, q7, q8], start=1):
    print(f"Q{i}:", answer)

## 20. Final mental model

Whenever you need a seasonal period:

1. Identify how often the data is observed (the frequency).
2. Identify the cycle you expect to repeat, and why it repeats.
3. Count how many observations occur in that cycle: that is $m$.
4. Check you have at least $2m$ observations.
5. Confirm it with a seasonal plot and lag correlations (after removing trend).
6. Decide additive or multiplicative.
7. Handle what $m$ cannot capture: moving holidays need event features; non-integer or very long periods need Fourier terms.
8. Benchmark against seasonal naive before trusting any model.

Example: hourly data with weekly seasonality:

$$
24\times7=168
$$

Therefore:

$$
\boxed{m=168}
$$

> **Frequency tells you how often observations are recorded. Seasonal period tells you how many observations occur before a seasonal pattern repeats.**